In [25]:
pip install -q transformers datasets evaluate torch langchain langchain-core langchain-community faiss-cpu langchain-huggingface sentence-transformers

Note: you may need to restart the kernel to use updated packages.


# Dataset Formation and making the corpus

In [26]:
from datasets import load_dataset
data = load_dataset('hotpotqa/hotpot_qa','distractor')

KeyboardInterrupt: 

In [ ]:
data['train']

In [ ]:
# data['train'][0]

In [34]:
data['train'][0]['context']['title']

['Radio City (Indian radio station)',
 'History of Albanian football',
 'Echosmith',
 "Women's colleges in the Southern United States",
 'First Arthur County Courthouse and Jail',
 "Arthur's Magazine",
 '2014–15 Ukrainian Hockey Championship',
 'First for Women',
 'Freeway Complex Fire',
 'William Rast']

In [35]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

In [ ]:
corpus = {}
for datapoint in data['train']:
    titles  = datapoint['context']['title']
    sentences_list = datapoint['context']['sentences']
    for title,sentences_list in zip(titles,sentences_list):
        text = " ".join(sentences_list) 
        corpus[title] = {
            "title": title,
            "text": text
        }
    

In [ ]:
len(corpus)

In [ ]:
print(next(iter(corpus)))

In [ ]:
print(data["train"][0]["context"]["title"][:5])
print(data["train"][1]["context"]["title"][:5])
print(data["train"][2]["context"]["title"][:5])

In [ ]:
print(len(set(corpus.keys())))
print(len(corpus))

# Chunking

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

In [ ]:
lengths = []

for article in corpus.values():
    text = " ".join(article["text"].split())
    token_count = len(tokenizer.tokenize(text))
    lengths.append(token_count)

In [ ]:
print(f"Number of articles : {len(lengths)}")
print(f"Minimum tokens     : {min(lengths)}")
print(f"Maximum tokens     : {max(lengths)}")
print(f"Average tokens     : {sum(lengths)/len(lengths):.2f}")

In [ ]:
thresholds = [128, 256, 384, 512]

for t in thresholds:
    count = sum(length > t for length in lengths)
    print(f">{t} tokens : {count} ({count/len(lengths)*100:.2f}%)")

In [89]:
# converting the corpus into documents
from langchain_core.documents import Document
documents = []

for article in corpus.values():
    text = " ".join(article["text"].split())

    documents.append(
        Document(
            page_content=f"Title: {article['title']}\n\nContent:\n{text}",
            metadata={
                "title": article["title"]
            }
        )
    )

In [90]:
print(documents[0].page_content)
print("=" * 80)
print(documents[1].page_content)
print("=" * 80)
print(documents[2].page_content)

Title: Radio City (Indian radio station)

Content:
Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title: History of Albanian football

Content:
Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which st

In [87]:
len(documents)

482021

# Embedding Model and embedding the documents

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [71]:
embedding = HuggingFaceEmbeddings(
    model_name= 'BAAI/bge-base-en-v1.5'
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from langchain_community.vectorstores import FAISS

In [91]:
batch_size = 1000

vector_store = FAISS.from_documents(
    documents[:batch_size],
    embedding = embedding
)

In [92]:
next_milestone = 100000

for i in range(batch_size, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    vector_store.add_documents(batch)

    indexed = min(i + batch_size, len(documents))
    while indexed >= next_milestone:
        print(f"✅ Indexed {next_milestone:,} documents")
        next_milestone += 100000

print(f"Finished indexing all {len(documents):,} documents")

✅ Indexed 100,000 documents
✅ Indexed 200,000 documents
✅ Indexed 300,000 documents
✅ Indexed 400,000 documents
Finished indexing all 482,021 documents


In [94]:
vector_store.save_local("hotpotqa_faiss")

In [95]:
vector_store = FAISS.load_local(
    "hotpotqa_faiss",
    embedding,
    allow_dangerous_deserialization=True
)

In [106]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs={
    "k": 10 ,
    } 
)


In [107]:
docs = retriever.invoke("Which magazine was started first Arthur's Magazine or First for Women?")
docs

[Document(id='2976cc30-942a-4521-a4d6-9e99957e86dc', metadata={'title': "Arthur's Lady's Home Magazine"}, page_content='Title: Arthur\'s Lady\'s Home Magazine\n\nContent:\nArthur\'s Home Magazine (1852-ca.1898) or Ladies\' Home Magazine was an American periodical published in Philadelphia by Timothy Shay Arthur. Editors Arthur and Virginia Francis Townsend selected writing and illustrations intended to appeal to female readers. Among the contributors: Mary Tyler Peabody Mann and Kate Sutherland. In its early years the monthly comprised a selection of articles originally published in Arthur\'s weekly "Home Gazette." Its nonfiction stories contained occasional factual inaccuracies for the sake of a good read. A contemporary review judged it "gotten up in good taste and well; and is in nothing overdone. Even its fashion plates are not quite such extravagant caricatures of rag-baby work as are usually met with in some of the more fancy magazines." Readers included patrons of the Mercantile

In [101]:
data['train'][0]["supporting_facts"]["title"]

["Arthur's Magazine", 'First for Women']

Function to check how accurate the answers are getting retrieved

In [102]:
def evaluate_retriever(example, retriever):
    question = example["question"]

    gold_titles = set(example["supporting_facts"]["title"])

    retrieved_docs = retriever.invoke(question)

    retrieved_titles = {
        doc.metadata["title"]
        for doc in retrieved_docs
    }

    correct = len(gold_titles & retrieved_titles)
    recall = correct / len(gold_titles)

    return recall

In [103]:
evaluate_retriever(data["train"][0], retriever)

1.0

In [105]:
# score for the first 1000 documents
scores = []


for k in [3, 5, 10, 20]:
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k,
            # "fetch_k": max(20, 2 * k),
            
        }
    )

    scores = []

    for example in data["train"].select(range(1000)):
        scores.append(evaluate_retriever(example, retriever))

    print(f"Recall@{k}: {sum(scores)/len(scores):.4f}")

Recall@3: 0.7055
Recall@5: 0.7540
Recall@10: 0.8050
Recall@20: 0.8355
